In [ ]:
# STEP 1: Load and Preprocess the Dataset

import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.corpus import stopwords


nltk.download('stopwords')

# Load dataset
df = pd.read_csv("NLP_Abstract_Dataset (Discipline)(105).csv")

# Basic cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

# Apply cleaning
df['Clean_Abstract'] = df['Abstract'].apply(clean_text)

# Encode labels
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Discipline'])

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    df['Clean_Abstract'], df['Label'], test_size=0.2, random_state=42, stratify=df['Label']
)

# Quick check
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

In [ ]:
# STEP 2: TF-IDF Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# ✅ Updated vectorizer with bigrams
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1,2), max_df=0.9, min_df=1)

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform the test data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Save the updated vectorizer
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')

# Sanity check
print("TF-IDF Vectorization complete with bigrams.")
print("Training shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

In [ ]:
# STEP 3: Train & Evaluate Models

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Logistic Regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_tfidf, y_train)
y_pred_logreg = logreg.predict(X_test_tfidf)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)

# Evaluation
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_logreg))
print("\nLogistic Regression Report:\n", classification_report(y_test, y_pred_logreg))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_logreg))

print("\n===============================\n")

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nNaive Bayes Report:\n", classification_report(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

### Model Selection Rationale

🧠 Model Selection Rationale – Discipline Classifier

Both Logistic Regression and Naive Bayes classifiers were evaluated on the task of classifying 105 research paper abstracts into one of three computing disciplines: CS (0), IS (1), and IT (2).

Although both models performed well, **Logistic Regression was selected as the final model** based on the following:

- ✅ **Logistic Regression achieved 90.48% accuracy** on the test set (19/21 correct), outperforming Naive Bayes (85.71%)
- ✅ It showed **perfect classification of CS abstracts** (Precision = 1.00, Recall = 1.00, F1 = 1.00 for class 0)
- ✅ Only two total misclassifications occurred (IS ↔ IT), compared to three in Naive Bayes
- ✅ Logistic Regression yielded higher macro and weighted F1-scores (0.90 vs. 0.87) and better overall calibration
- ✅ It offers better model interpretability, making it more suitable for future extensions to subfield and methodology classification

Thus, Logistic Regression was chosen as the most reliable and generalizable model for the discipline-level classification task. The final model and vectorizer were saved as:

- `discipline_classifier_logreg.pkl`
- `tfidf_vectorizer.pkl`

In [ ]:
# STEP 4: Save the final model
import joblib

# Save Logistic Regression model to disk
joblib.dump(logreg, "discipline_classifier_logreg.pkl")

print("Final Logistic Regression model saved as discipline_classifier_logreg.pkl")